
Ce carnet utilise **ollama** avec le modèle `gemma3:4b` pour déterminer le genre associé au nom de l'hôte (`reviewer_name`). Les catégories possibles sont : **homme**, **femme**, **couple**, **épicène**, et **entreprise**. Nous travaillerons initialement sur les 20 premières lignes du fichier `reviews_select.csv`.


Configurer l'environnement pour exécuter des commandes système et s'assurer que les chemins sont corrects.

In [ ]:

# Définir le répertoire de travail, si nécessaire
base_dir = os.getcwd()
print(f"Working directory: {base_dir}")


Importer les bibliothèques nécessaires : pandas pour la manipulation de données et subprocess pour appeler Ollama.

In [ ]:
import subprocess

# Vérifier qu'ollama est accessible
!ollama --help


Charger les 20 premières entrées du fichier CSV.

In [ ]:
df = pd.read_csv(csv_path, nrows=20)
df.head()


Aucun nettoyage majeur n'est prévu pour ce jeu de données restreint. Nous inspectons simplement les colonnes pertinentes.


Examiner rapidement les noms et les commentaires pour avoir un aperçu.

In [ ]:
    print(i, row['reviewer_name'], '-', row['comments'][:60])


Pas de visualisation nécessaire pour cette tâche ponctuelle.


Classifier chaque enregistrement via Ollama et sauvegarder les résultats.

In [ ]:

def classify(name, comment):
    prompt = (
        f"Vous êtes un assistant. Classez le nom de l'hôte (reviewer_name) dans une des catégories suivantes : {', '.join(categories)}.\n"
        f"N'utilisez que le nom et éventuellement le commentaire pour déterminer le genre.\n"
        f"Répondez uniquement par un mot parmi les catégories, sans autre texte.\n"
        f"Nom : {name}\nCommentaire : {comment}\n"
    )
    proc = subprocess.run(
        ['ollama', 'run', 'gemma3:4b'],
        input=prompt.encode('utf-8'),
        stdout=subprocess.PIPE,
    )
    out = proc.stdout.decode('utf-8').strip().splitlines()[0] if proc.stdout else ''
    return out

# Appliquer la classification
results = []
for idx, row in df.iterrows():
    gender = classify(row['reviewer_name'], row.get('comments', ''))
    results.append(gender)

df['gender'] = results
out = df[['reviewer_name', 'comments', 'gender']]
out.to_csv('first20_gender.csv', index=False)
out
